### Plain Numpy implementation with no autograd

In [1]:
import numpy as np 

# f = w * x 
# f = 2 * x 

X = np.array([1, 2, 3, 4], dtype = np.float32)
Y = np.array([2, 4, 6, 8], dtype=np.float32)

w = 0.0

# model prediction 
def forward(x): 
    return w * x

# loss = MSE
def loss(y, y_predicted):
    return ((y_predicted - y)**2).mean()

# gradient 
# MSE formula: = 1/N * (w*x - y)**2
# MSE derivative: dJ/dw = 1/N * 2x(w*x - y)
def gradient(x, y, y_predicted): 
    return np.dot(2*x, y_predicted - y).mean()

print(f"Prediction before training: f(5) = {forward(5):.3f}")

# Training 
learning_rate = .01
n_iters = 20 

for epoch in range(n_iters): 
    # prediction = forward pass 
    y_pred = forward(X) 

    # loss 
    l = loss(Y, y_pred) 

    # gradients 
    dw = gradient(X, Y, y_pred) 

    # update weights 
    w -= learning_rate * dw 

    if epoch % 2 == 0: 
        print(f'epoch {epoch+1}: w = {w:.3f}, loss = {l:.8f}')

print(f"Prediction after training: f(5) = {forward(5):.3f}")

Prediction before training: f(5) = 0.000
epoch 1: w = 1.200, loss = 30.00000000
epoch 3: w = 1.872, loss = 0.76800019
epoch 5: w = 1.980, loss = 0.01966083
epoch 7: w = 1.997, loss = 0.00050332
epoch 9: w = 1.999, loss = 0.00001288
epoch 11: w = 2.000, loss = 0.00000033
epoch 13: w = 2.000, loss = 0.00000001
epoch 15: w = 2.000, loss = 0.00000000
epoch 17: w = 2.000, loss = 0.00000000
epoch 19: w = 2.000, loss = 0.00000000
Prediction after training: f(5) = 10.000


### Pytorch low level with autograd

In [7]:
import torch 

# f = w * x 
# f = 2 * x 

X = torch.tensor([1, 2, 3, 4], dtype = torch.float32)
Y = torch.tensor([2, 4, 6, 8], dtype=torch.float32)

# interested in grad of loss with respect to w => requires grad
w = torch.tensor(0.0, dtype=torch.float32, requires_grad=True)

# model prediction - same
def forward(x): 
    return w * x

# loss = MSE - same 
def loss(y, y_predicted):
    return ((y_predicted - y)**2).mean()

# gradient - computed automatically by torch here

print(f"Prediction before training: f(5) = {forward(5):.3f}")

# Training, autograd not as precise as manual gradient computation
learning_rate = .01
n_iters = 100 

for epoch in range(n_iters): 
    # prediction = forward pass 
    y_pred = forward(X) 

    # loss 
    l = loss(Y, y_pred) 

    # gradients = backward pass
    l.backward() # dl/dw (only w has grad)

    # update weights, this operation should not be part in the comp graph
    with torch.no_grad():
        w -= learning_rate * w.grad 

    # zero gradients accumulated in this iteration for next iteration
    w.grad.zero_()

    if epoch % 10 == 0: 
        print(f'epoch {epoch+1}: w = {w:.3f}, loss = {l:.8f}')

print(f"Prediction after training: f(5) = {forward(5):.3f}")

Prediction before training: f(5) = 0.000
epoch 1: w = 0.300, loss = 30.00000000
epoch 11: w = 1.665, loss = 1.16278565
epoch 21: w = 1.934, loss = 0.04506890
epoch 31: w = 1.987, loss = 0.00174685
epoch 41: w = 1.997, loss = 0.00006770
epoch 51: w = 1.999, loss = 0.00000262
epoch 61: w = 2.000, loss = 0.00000010
epoch 71: w = 2.000, loss = 0.00000000
epoch 81: w = 2.000, loss = 0.00000000
epoch 91: w = 2.000, loss = 0.00000000
Prediction after training: f(5) = 10.000


### Pytorch loss / optimizer, but weights still manual

In [14]:
import torch 
import torch.nn as nn

# f = w * x 
# f = 2 * x 

X = torch.tensor([1, 2, 3, 4], dtype = torch.float32)
Y = torch.tensor([2, 4, 6, 8], dtype=torch.float32)

# interested in grad of loss with respect to w => requires grad
w = torch.tensor(0.0, dtype=torch.float32, requires_grad=True)

# model prediction - same
def forward(x): 
    return w * x

learning_rate = 0.01
n_iters = 100

# loss = MSE - same 
loss = nn.MSELoss()

# stochastic gradient descent - optimizes the weights
optimizer = torch.optim.SGD([w], lr=learning_rate)


print(f"Prediction before training: f(5) = {forward(5):.3f}")

# Training, autograd not as precise as manual gradient computation
learning_rate = .01
n_iters = 100 

for epoch in range(n_iters): 
    # prediction = forward pass 
    y_pred = forward(X) 

    # loss - callable function
    l = loss(Y, y_pred) 

    # gradients = backward pass
    l.backward() # dl/dw (only w has grad)

    # optimizer updates steps automatically 
    optimizer.step()

    # still have to empty gradients 
    optimizer.zero_grad()

    if epoch % 10 == 0: 
        print(f'epoch {epoch+1}: w = {w:.3f}, loss = {l:.8f}')

print(f"Prediction after training: f(5) = {forward(5):.3f}")

Prediction before training: f(5) = 0.000
epoch 1: w = 0.300, loss = 30.00000000
epoch 11: w = 1.665, loss = 1.16278565
epoch 21: w = 1.934, loss = 0.04506890
epoch 31: w = 1.987, loss = 0.00174685
epoch 41: w = 1.997, loss = 0.00006770
epoch 51: w = 1.999, loss = 0.00000262
epoch 61: w = 2.000, loss = 0.00000010
epoch 71: w = 2.000, loss = 0.00000000
epoch 81: w = 2.000, loss = 0.00000000
epoch 91: w = 2.000, loss = 0.00000000
Prediction after training: f(5) = 10.000


### Pytorch model automatically defines weights & prediction func

In [10]:
import torch 

# f = w * x 
# f = 2 * x 

# num of rows = num samples, each row contains the features of one sample
X = torch.tensor([[1], [2], [3], [4]], dtype = torch.float32)
Y = torch.tensor([[2], [4], [6], [8]], dtype=torch.float32)

# test also has to be a tensor 
X_test = torch.tensor([5], dtype=torch.float32)

n_samples, n_features = X.shape
print(f'#samples: {n_samples}, #features: {n_features} for each sample')

input_size = n_features # 1
output_size = n_features # 1

## Custom model
class LinearRegression(nn.Module): 
    def __init__(self, input_dim, output_dim): 
        super(LinearRegression, self).__init__()

        # define layers
        self.lin = nn.Linear(input_dim, output_dim) 

    def forward(self, x): 
        return self.lin(x) 
    
# weights, prediction func, loss func defined by torch 
model = LinearRegression(input_size, output_size)


# gradient - computed automatically by torch here

print(f"Prediction before training: f(5) = {model(X_test).item():.3f}")

# Training, autograd not as precise as manual gradient computation
learning_rate = .01
n_iters = 100 

loss = torch.nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(), lr = learning_rate)

for epoch in range(n_iters): 
    # prediction = forward pass 
    y_pred = model(X) 

    # loss 
    l = loss(Y, y_pred) 

    # gradients = backward pass
    l.backward() # dl/dw (only w has grad)

    # update weights via optimizer directly
    optimizer.step()

    # zero gradients accumulated in this iteration for next iteration
    optimizer.zero_grad()

    if epoch % 10 == 0: 
        [w, b] = model.parameters() # list of lists
        print(f'epoch {epoch+1}: w = {w[0][0].item():.3f}, loss = {l:.8f}')

print(f"Prediction after training: f(5) = {model(X_test).item():.3f}")

#samples: 4, #features: 1 for each sample
Prediction before training: f(5) = 4.238
epoch 1: w = 0.861, loss = 8.45098686
epoch 11: w = 1.530, loss = 0.38523552
epoch 21: w = 1.646, loss = 0.16685894
epoch 31: w = 1.673, loss = 0.15207708
epoch 41: w = 1.686, loss = 0.14309427
epoch 51: w = 1.695, loss = 0.13476208
epoch 61: w = 1.704, loss = 0.12691824
epoch 71: w = 1.713, loss = 0.11953093
epoch 81: w = 1.722, loss = 0.11257359
epoch 91: w = 1.730, loss = 0.10602113
Prediction after training: f(5) = 9.458
